# Reopen a Clerk case: Atera human colorectal cancer

Scratch notebook (not part of the docs) for reading back through a saved Clerk case, like a record you scroll through: its questions, answers, findings and evidence, with the views it looked at.

`dega.viz.Clerk(clerk_files=...)` reopens a case: a report folder (from **Report** or `clerk.export()`), a saved conversation (**Save**), or a folder of them such as `clerk_files`, which opens the most recent. **No Claude needed**:

- the panel shows the whole conversation; its genes, clusters and cited views change the linked widgets below;
- each question's views are pins on the Landscape (**PINS**): hover one for its question and a preview, click it to return to that view;
- asking a new question continues the case (that does need Claude), and new reports are saved beside it.

The links need the same data loaded, so this opens the case on the data of `Clerk_Atera_CRC_Landscape-Clustergram-Yearbook.ipynb`, which saves its cases to `clerk_files/atera_crc/` next to it.

In [ ]:
import os

# Use this checkout's JS build, served locally (restart the kernel after a rebuild).
os.environ["CELLDEGA_LOCAL_ESM"] = "serve"

In [ ]:
from pathlib import Path
import tempfile
from urllib.request import urlopen

import celldega as dega
import numpy as np
import zarr

SAMPLE = "Atera_v1_WTA_S4B_Colorectal_Cancer_FF"
HF_BASE = "https://huggingface.co/datasets/broadinstitute/Celldega_Supporting_Data/resolve/main/Atera"
BASE_URL = "https://raw.githubusercontent.com/broadinstitute/celldega_atera-wta-ff-human-crc/main"
MATRIX_URL = f"{HF_BASE}/{SAMPLE}/cgm/default_clusters"
DATA_DIR = Path(tempfile.gettempdir()) / "celldega_atera" / SAMPLE
DATA_DIR.mkdir(parents=True, exist_ok=True)

# This dataset's saved cases (the CRC notebook saves them here): the folder opens the
# most recent; point at one case's folder to open that one.
CLERK_FILES = Path("clerk_files/atera_crc")

with urlopen(f"{BASE_URL}/micron_to_image_transform.csv") as response:
    transform = np.loadtxt(response)

local = DATA_DIR / "default_clusters.zarr.zip"
if not local.is_file():
    with urlopen(f"{MATRIX_URL}.zarr.zip") as response:
        local.write_bytes(response.read())
with zarr.storage.ZipStore(str(local), mode="r") as store:
    matrix = dega.clust.Matrix.read_zarr(store)

sorted(p.parent.name if p.name == "record.json" else p.name
       for p in [*CLERK_FILES.glob("*/record.json"), *CLERK_FILES.glob("conversations/*.json")])

## The case, on its data

The widgets are made first and passed to Clerk, so the case's views become pins on the Landscape.

In [ ]:
clustergram = dega.viz.Clustergram(
    matrix=matrix, width=400, height=500, rank_dim=3,
    viz_mode="dotplot", manual_row_cat=False, manual_col_cat=False,
)
landscape = dega.viz.Landscape(
    technology="Xenium", base_url=BASE_URL, transform=transform,
    dataset_name="Atera colorectal cancer — default clusters", height=600,
    max_tiles_to_view=25,
)
yearbook = dega.viz.Yearbook(
    base_url=BASE_URL,
    front_end_query={"cluster": {"attr": "cluster", "value": "3"}, "gene": "CLDN4", "max_cells": 60},
    rows=2, cols=6, portrait_size_um=50,
    dataset_name="Atera colorectal cancer — cell yearbook", width=0, height=520,
)
yearbook.layout.width = "100%"

clerk = dega.viz.Clerk(
    clerk_files=CLERK_FILES, landscape=landscape, clustergram=clustergram, yearbook=yearbook
)
print("Opened", clerk._case["path"], "saved", clerk._case.get("saved"))
print(len(landscape.pins), "pins")

In [ ]:
dega.viz.spatial_clustergram(landscape, clustergram, width="520px", clerk=clerk)

In [ ]:
yearbook

## Findings

The cell types Clerk proposed in the case, with their evidence:

In [ ]:
clerk.findings_df()

Its views as Snapshot cards, by question (e.g. `"Q1 Landscape"`):

In [ ]:
clerk.snapshots

In [ ]:
# clerk.snapshots.card("Q1 Landscape", landscape=landscape)